In [7]:
import h5py
import random
import numpy as np
import matplotlib.pyplot as plt

from torch.utils.data import Dataset, DataLoader

import os
import numpy as np

import torch
import torch.optim as optim
from torch import nn
from torch.utils.data import DataLoader
import torch.nn.functional as F
import torchvision.models as models
import torchvision.transforms as transforms

In [ ]:
class LystoDataset(Dataset):

    def __init__(self, filepath=None,
                 transform=None,
                 train=True,
                 kfold=5):

        if filepath:
            f = h5py.File(filepath, 'r')
        else:
            raise Exception("Invalid data file.")

        if kfold <= 0:
            raise Exception("Invalid k-fold cross-validation argument.")

        self.train = train      # 训练集 / 验证集
        self.kfold = kfold      # k 折交叉验证
        self.organs = []        # 全切片来源，array (20000)
        self.images = []        # array ( 20000 * 299 * 299 * 3 )
        self.labels = []        # 图像中的阳性细胞数目，array ( 20000 )
        self.imageIDX = []      # 每个patch对应的图像号，array ( 20000 * n )
        self.patches = []       # 每张图像中选取的像素 patch 的左上角坐标点，array ( 20000 * n * 2 )
        self.interval = 50       # 取实例的像素间隔
        self.size = 32          # 一个实例的大小

        imageIDX = -1
        for i, (organ, img, label) in enumerate(zip(f['organ'], f['x'], f['y'])):

            if i == 51:
                break

            if (self.train and (i + 1) % self.kfold == 0) or (not self.train and (i + 1) % self.kfold != 0):
                continue

            imageIDX += 1
            self.organs.append(organ)
            self.images.append(img)
            # self.labels.append(label)
            self.labels.append(1 if label != 0 else 0) # TODO: 暂时把标签当作非计数式标签处理
            p = get_patches(img, self.interval, self.size)
            self.patches.append(p) # 获取 32 * 32 的实例
            self.imageIDX.extend([imageIDX] * len(p))

        self.mode = None
        self.transform = transform

    def setmode(self, mode):
        self.mode = mode

    def make_train_data(self, idxs, shuffle=True): # 用于 mode 2，制作训练用数据集
        self.train_data = [(self.imageIDX[i], self.patches[i // len(self.patches[0])][i % len(self.patches[0])],
                            self.labels[self.imageIDX[i]]) for i in idxs]
        if shuffle:
            self.train_data = random.sample(self.train_data, len(self.train_data))

    def __getitem__(self, idx):

        # organ = self.organs[idx]

        if self.mode == 1: # top-k 选取模式
            (x, y) = self.patches[idx // len(self.patches[0])][idx % len(self.patches[0])]
            patch = self.images[self.imageIDX[idx]][x:x + self.size - 1, y:y + self.size - 1]
            if self.transform is not None:
                patch = self.transform(patch)

            label = self.labels[self.imageIDX[idx]]

        elif self.mode == 2: # 训练数据模式
            imageIDX, patch_grid, label = self.train_data[idx]
            # print("imageIDX:{}".format(imageIDX))
            # print("patch_grid:{}".format(patch_grid))
            # print("label:{}".format(label))
            (x, y) = patch_grid
            patch = self.images[imageIDX][x:x + self.size - 1, y:y + self.size - 1]
            if self.transform is not None:
                patch = self.transform(patch)

        else:
            raise Exception("Something wrong in setmode.")

        return patch, label

    def __len__(self):
        if self.mode == 1:
            return len(self.imageIDX)
        elif self.mode == 2:
            return len(self.train_data)
        else:
            raise Exception("Something wrong in setmode.")

def get_patches(image, interval=3, size=32):
    """
    在每张图片上生成小patch实例。
    :param image: 输入图片矩阵，299 x 299 x 3
    :param interval: 取patch坐标点的间隔
    :param size: 单个patch的大小
    """

    patches = []
    for x in np.arange(0, image.shape[0] - size + 1, interval):
        for y in np.arange(0, image.shape[1] - size + 1, interval):
            patches.append((x, y))  # n x 2

    # for i, (x, y) in enumerate(patches):
    #     patches[i] = image[x:x+size-1, y:y+size-1]  # n x 32 x 32 x 3

    return patches

In [50]:
max_acc = 0
def train(trainset, valset, batch_size, total_epochs, test_every, model, criterion, optimizer, topk, output_path):
    global max_acc

    train_loader = DataLoader(trainset, batch_size=batch_size, shuffle=False)
    val_loader = DataLoader(valset, batch_size=batch_size, shuffle=False)

    print('Start training ...')

    for epoch in range(1, total_epochs + 1):

        trainset.setmode(1)

        # 获取实例分类概率
        model.eval()
        probs = torch.FloatTensor(len(train_loader.dataset))
        with torch.no_grad():
            # 禁止反向传播
            for i, input in enumerate(train_loader):
                print('Inference\tEpoch: [{}/{}]\tBatch: [{}/{}]'
                      .format(epoch, total_epochs, i + 1, len(train_loader)))
                # softmax 输出[[a,b],[c,d]] shape = batch_size*2
                output = F.softmax(model(input[0]), dim=1)
                # detach()[:,1]取出softmax得到的概率，产生：[b, d, ...]
                # input.size(0)返回batch中的实例数量
                probs[i * batch_size:i * batch_size + input[0].size(0)] = output.detach()[:, 1].clone()

        # 找出top-k
        probs = probs.numpy()
        groups = np.array(trainset.imageIDX)
        order = np.lexsort((probs, groups))
        groups = groups[order]
        # probs = probs[order]
        index = np.empty(len(groups), 'bool')
        index[-topk:] = True
        # 同时把属于每个slide的、pred最大的k个实例挑出来，放入topk中
        index[:-topk] = groups[topk:] != groups[:-topk]

        # 根据top-k的分类，制作迭代使用的数据集
        trainset.make_train_data(list(order[index]))

        trainset.setmode(2)

        # 训练
        model.train()
        train_loss = 0.
        for i, (data, label) in enumerate(train_loader):
            output = model(data)
            # reset gradients
            optimizer.zero_grad()
            # calculate loss
            loss = criterion(output, label)
            train_loss += loss.item() * data.size(0)

            # backward pass
            loss.backward()
            # step
            optimizer.step()

        # calculate loss and error for epoch
        train_loss /= len(train_loader)
        print('Epoch: [{}/{}], Loss: {:.4f}'.format(epoch, total_epochs, train_loss))

        # 验证

        if (epoch + 1) % test_every == 0:

            print('\nValidating ...')

            valset.setmode(1)
            model.eval()
            val_probs = torch.FloatTensor(len(val_loader.dataset))
            with torch.no_grad():
                # 禁止反向传播
                for i, input in enumerate(val_loader):
                    print('Inference\tEpoch: [{}/{}]\tBatch: [{}/{}]'
                          .format(epoch, total_epochs, i + 1, len(val_loader)))
                    # 把训练过的 model 在验证集上做一下
                    val_output = F.softmax(model(input[0]), dim=1)
                    val_probs[i * batch_size:i * batch_size + input[0].size(0)] \
                        = val_output.detach()[:, 1].clone()

            val_probs = val_probs.numpy()
            val_groups = np.array(valset.imageIDX)

            # TODO: 暂时把标签当作非计数式标签处理
            max_prob = np.empty(len(valset.labels))  # 模型预测的实例最大概率列表，每张切片取最大概率的 patch
            max_prob[:] = np.nan
            order = np.lexsort((val_probs, val_groups))
            # 排序
            val_groups = val_groups[order]
            val_probs = val_probs[order]
            # 取最大
            val_index = np.empty(len(val_groups), 'bool')
            val_index[-1] = True
            val_index[:-1] = val_groups[1:] != val_groups[:-1]
            max_prob[val_groups[val_index]] = val_probs[val_index]

            pred = [1 if prob >= 0.5 else 0 for prob in max_prob] # 每张切片由最大概率的 patch 得到的标签
            err, fpr, fnr = calc_err(pred, valset.labels)
            # 计算
            print('Validation\tEpoch: [{}/{}]\tError: {}\tFPR: {}\tFNR: {}\n'
                  .format(epoch, total_epochs, err, fpr, fnr))

            # Save the best model
            acc = 1 - (fpr + fnr) / 2.
            if acc >= max_acc:
                max_acc = acc
                obj = {
                    'epoch': epoch + 1,
                    'state_dict': model.state_dict(),
                    'max_accuracy': max_acc,
                    'optimizer': optimizer.state_dict()
                }
                torch.save(obj, os.path.join(output_path, 'checkpoint_best.pth'))

def calc_err(pred, real):
    pred = np.array(pred)
    real = np.array(real)
    neq = np.not_equal(pred, real)
    err = float(neq.sum())/pred.shape[0]
    fpr = float(np.logical_and(pred==1,neq).sum())/(real==0).sum()
    fnr = float(np.logical_and(pred==0,neq).sum())/(real==1).sum()
    return err, fpr, fnr

In [ ]:
# 调试train()
torch.manual_seed(1)

print('Init Model ...')
model = models.resnet18(pretrained=True)
model.fc = nn.Linear(model.fc.in_features, 2)

normalize = transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
trans = transforms.Compose([transforms.ToTensor(), normalize]) # TODO: 设计归一化方式

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.0005, weight_decay=1e-4)

print('Loading Dataset ...')

imageSet = LystoDataset(filepath = "D:/LYSTO/training.h5", transform = trans)
imageSet_val = LystoDataset(filepath = "D:/LYSTO/training.h5", transform = trans, train = False)

train_loader = DataLoader(imageSet, batch_size=2, shuffle=False)
val_loader = DataLoader(imageSet_val, batch_size=2, shuffle=False)

train(imageSet, imageSet_val, 2, 5, 5, model, criterion, optimizer, 2, "D:")


Init Model ...
Loading Dataset ...
Start training ...
Inference	Epoch: [1/5]	Batch: [1/29525]
Inference	Epoch: [1/5]	Batch: [2/29525]
Inference	Epoch: [1/5]	Batch: [3/29525]
Inference	Epoch: [1/5]	Batch: [4/29525]
Inference	Epoch: [1/5]	Batch: [5/29525]
Inference	Epoch: [1/5]	Batch: [6/29525]
Inference	Epoch: [1/5]	Batch: [7/29525]
Inference	Epoch: [1/5]	Batch: [8/29525]
Inference	Epoch: [1/5]	Batch: [9/29525]
Inference	Epoch: [1/5]	Batch: [10/29525]
Inference	Epoch: [1/5]	Batch: [11/29525]
Inference	Epoch: [1/5]	Batch: [12/29525]
Inference	Epoch: [1/5]	Batch: [13/29525]
Inference	Epoch: [1/5]	Batch: [14/29525]
Inference	Epoch: [1/5]	Batch: [15/29525]
Inference	Epoch: [1/5]	Batch: [16/29525]
Inference	Epoch: [1/5]	Batch: [17/29525]
Inference	Epoch: [1/5]	Batch: [18/29525]
Inference	Epoch: [1/5]	Batch: [19/29525]
Inference	Epoch: [1/5]	Batch: [20/29525]
Inference	Epoch: [1/5]	Batch: [21/29525]
Inference	Epoch: [1/5]	Batch: [22/29525]
Inference	Epoch: [1/5]	Batch: [23/29525]
Inference	Ep

In [51]:
# 调试数据集
torch.manual_seed(1)

print('Init Model ...')
model = models.resnet18(pretrained=True)
model.fc = nn.Linear(model.fc.in_features, 2)

# normalize = transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.1, 0.1, 0.1])
# trans = transforms.Compose([transforms.ToTensor(), normalize])
trans = transforms.ToTensor()

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.0005, weight_decay=1e-4)

print('Loading Dataset ...')
imageSet = LystoDataset(filepath = "D:/LYSTO/training.h5", transform = trans)
imageSet_val = LystoDataset(filepath = "D:/LYSTO/training.h5", transform = trans, train = False)

train_loader = DataLoader(imageSet, batch_size=2, shuffle=False)
val_loader = DataLoader(imageSet_val, batch_size=2, shuffle=False)

imageSet.setmode(1)
imageSet_val.setmode(1)
for idx, data in enumerate(train_loader):
    print('Dry Run : [{}/{}]\r'.format(idx + 1, len(train_loader.dataset) // 2))
print("Length of dataset: {}".format(len(train_loader.dataset)))
for idx, data in enumerate(val_loader):
    print('Dry Run : [{}/{}]\r'.format(idx + 1, len(val_loader.dataset) // 2))
print("Length of dataset: {}".format(len(val_loader.dataset)))

Init Model ...
Loading Dataset ...
Dry Run : [1/450]
Dry Run : [2/450]
Dry Run : [3/450]
Dry Run : [4/450]
Dry Run : [5/450]
Dry Run : [6/450]
Dry Run : [7/450]
Dry Run : [8/450]
Dry Run : [9/450]
Dry Run : [10/450]
Dry Run : [11/450]
Dry Run : [12/450]
Dry Run : [13/450]
Dry Run : [14/450]
Dry Run : [15/450]
Dry Run : [16/450]
Dry Run : [17/450]
Dry Run : [18/450]
Dry Run : [19/450]
Dry Run : [20/450]
Dry Run : [21/450]
Dry Run : [22/450]
Dry Run : [23/450]
Dry Run : [24/450]
Dry Run : [25/450]
Dry Run : [26/450]
Dry Run : [27/450]
Dry Run : [28/450]
Dry Run : [29/450]
Dry Run : [30/450]
Dry Run : [31/450]
Dry Run : [32/450]
Dry Run : [33/450]
Dry Run : [34/450]
Dry Run : [35/450]
Dry Run : [36/450]
Dry Run : [37/450]
Dry Run : [38/450]
Dry Run : [39/450]
Dry Run : [40/450]
Dry Run : [41/450]
Dry Run : [42/450]
Dry Run : [43/450]
Dry Run : [44/450]
Dry Run : [45/450]
Dry Run : [46/450]
Dry Run : [47/450]
Dry Run : [48/450]
Dry Run : [49/450]
Dry Run : [50/450]
Dry Run : [51/450]
Dry R